# Lembar Kerja Mahasiswa (LKM)
## Praktikum Pertemuan 04 — Unsupervised Learning dan Probabilitas

**Cakupan materi:** Unsupervised Learning · Probabilitas · K-Means · Gaussian Mixture Model (GMM)

---

### Identitas Mahasiswa

*Klik dua kali sel ini untuk mengedit, isi titik-titik di bawah, lalu tekan `Shift + Enter`.*

| | |
|---|---|
| **Nama** | ............................................................ |
| **NIM** | ............................................................ |
| **Kelas / Rombel** | ............................................................ |
| **Nama Dosen** | ............................................................ |
| **Tanggal Praktikum** | ............................................................ |

---

### Capaian yang Diukur

Setelah menyelesaikan lembar kerja ini, Anda diharapkan mampu:

1. Menjelaskan perbedaan mendasar antara *supervised* dan *unsupervised learning*.
2. Menerapkan K-Means dengan scikit-learn dan menunjukkan kepekaannya terhadap inisialisasi.
3. Menuliskan sendiri tahap penugasan dan pembaruan pada algoritma Lloyd.
4. Memilih jumlah klaster dengan metode siku dan skor silhouette.
5. Membedakan *hard clustering* (K-Means) dari *soft clustering* (GMM) beserta konsekuensinya.
6. Menerapkan Teorema Bayes dan menafsirkan pengaruh nilai *prior* yang kecil.


---
## Petunjuk Pengerjaan

**Baca bagian ini sebelum mulai.**

1. Jalankan sel kode **berurutan dari atas ke bawah**. Banyak sel bergantung pada hasil sel sebelumnya.
2. Sel yang berisi tanda `____` atau komentar `# TODO` **harus Anda lengkapi sendiri**. Sel akan error bila dijalankan sebelum dilengkapi — itu normal.
3. Setiap kegiatan memiliki **tabel hasil pengamatan** dan **kotak jawaban** berupa sel markdown. Klik dua kali untuk mengedit, lalu tekan `Shift + Enter`.
4. Angka pada tabel hasil **harus berasal dari eksekusi di komputer Anda sendiri**.
5. Jawaban analisis dinilai dari **penalarannya**, bukan panjangnya. Dua sampai empat kalimat yang tepat lebih bernilai daripada satu paragraf mengambang.

> **Penting.** Nilai `random_state` diturunkan dari NIM Anda, sehingga hasil setiap mahasiswa akan sedikit berbeda. Jangan menyalin angka orang lain.

### Perkiraan waktu

| Bagian | Perkiraan |
|---|---|
| Persiapan | 5 menit |
| Kegiatan 1–3 (K-Means) | 45 menit |
| Kegiatan 4–5 (memilih k dan GMM) | 40 menit |
| Kegiatan 6 (Bayes) | 20 menit |
| Kesimpulan dan ekspor PDF | 10 menit |

### Cara menyimpan sebagai PDF

Setelah semua sel dijalankan dan seluruh jawaban terisi:

* **JupyterLab:** `File` → `Save and Export Notebook As...` → `HTML`, lalu buka berkas HTML di browser dan cetak (`Ctrl + P`) dengan tujuan **Save as PDF**.
* **Google Colab:** `File` → `Print` → tujuan **Save as PDF**.
* Beri nama berkas: `LKM04_NIM_NamaLengkap.pdf`

> Jalur lewat HTML dianjurkan karena ekspor PDF langsung membutuhkan LaTeX yang sering belum terpasang.


---
## Persiapan

Bagian ini **sudah lengkap** — Anda hanya perlu menjalankannya.

> **Catatan.** Seluruh data pada lembar kerja ini dibangkitkan sendiri atau berasal dari scikit-learn, sehingga **tidak memerlukan koneksi internet**.

### P.1 Memuat pustaka

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, load_wine
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams["figure.figsize"] = (6, 5)
print("Semua pustaka berhasil dimuat.")

### P.2 Mengisi NIM Anda

Ganti angka di bawah dengan **tiga digit terakhir NIM** Anda. Angka ini dipakai sebagai `random_state` pribadi.

In [ ]:
# TODO (a): ganti dengan tiga digit terakhir NIM Anda, misalnya 137
NIM_3_DIGIT = ____

SEED = int(NIM_3_DIGIT)
print("random_state pribadi Anda (SEED) =", SEED)

### P.3 Membangkitkan data

Kita memakai dua dataset buatan agar seluruh gejala yang ingin dipelajari terlihat jelas.

| Dataset | Bentuk gerombolan | Dipakai pada |
|---|---|---|
| `X_bulat` | Bulat dan terpisah jelas | Kegiatan 1–4 |
| `X_elips` | Memanjang dan miring | Kegiatan 5.1 |
| `X_tumpang` | Bulat tetapi saling bertumpang tindih | Kegiatan 5.2 |

> **Perhatikan.** Kita **menyimpan** label aslinya (`y_asli`) hanya untuk keperluan pemeriksaan di akhir. Label ini **tidak boleh** diberikan kepada algoritma — itulah inti *unsupervised learning*.

In [ ]:
# Dataset 1: empat gerombolan bulat yang terpisah jelas.
# Posisi pusatnya sengaja ditetapkan agar setiap mahasiswa memperoleh
# pola yang sebanding; SEED hanya memengaruhi deraunya.
PUSAT = np.array([[-6., -6.], [6., -6.], [-6., 6.], [6., 6.]])
X_bulat, y_asli = make_blobs(n_samples=400, centers=PUSAT, cluster_std=1.5,
                             random_state=SEED)

# Dataset 2: gerombolan yang sama, digeser sehingga menjadi memanjang dan miring
transformasi = np.array([[1.0, 0.0], [2.2, 0.35]])
X_elips = X_bulat @ transformasi

# Dataset 3: tiga gerombolan yang saling bertumpang tindih
X_tumpang, y_tumpang = make_blobs(n_samples=300,
                                  centers=np.array([[-2., 0.], [2., 0.], [0., 3.4]]),
                                  cluster_std=1.4, random_state=SEED)

print("Ukuran X_bulat  :", X_bulat.shape)
print("Ukuran X_elips  :", X_elips.shape)
print("Ukuran X_tumpang:", X_tumpang.shape)
print("Jumlah kelompok sebenarnya pada X_bulat:", len(np.unique(y_asli)))

---
# Kegiatan 1 — Mengenali Data Tanpa Label

**Rujukan:** bagian Unsupervised Learning pada modul Pertemuan 04.

**Pertanyaan yang ingin dijawab:** apa yang membedakan persoalan ini dari klasifikasi yang sudah kita pelajari pada Pertemuan 03?

### 1.1 Melihat datanya

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].scatter(X_bulat[:, 0], X_bulat[:, 1], s=18, c="gray", alpha=0.7)
ax[0].set_title("Yang KITA lihat (tanpa label)")
ax[0].set_xlabel("Fitur 1"); ax[0].set_ylabel("Fitur 2")

ax[1].scatter(X_bulat[:, 0], X_bulat[:, 1], s=18, c=y_asli, cmap="viridis", alpha=0.7)
ax[1].set_title("Kelompok sebenarnya (hanya untuk pemeriksaan)")
ax[1].set_xlabel("Fitur 1"); ax[1].set_ylabel("Fitur 2")

plt.tight_layout(); plt.show()

### 1.2 Tabel Hasil Pengamatan

| Aspek | Isian |
|---|---|
| Jumlah baris data | ...... |
| Jumlah fitur | ...... |
| Berapa gerombolan yang Anda lihat pada panel kiri? | ...... |
| Berapa gerombolan sebenarnya (panel kanan)? | ...... |

### 1.3 Pertanyaan Analisis

**A1.** Pada Pertemuan 03 kita melatih model dengan `model.fit(X, y)`. Pada modul ini kita menulis `model.fit(X)` saja. Apa arti hilangnya `y` bagi tugas yang harus dikerjakan algoritma?

> *Jawaban Anda:*
>
> ......

**A2.** Bila kita hanya melihat panel kiri, dari mana kita tahu bahwa jumlah kelompoknya empat dan bukan tiga atau lima? Apakah selalu ada jawaban yang pasti?

> *Jawaban Anda:*
>
> ......

**A3.** Sebutkan dua contoh persoalan nyata yang cocok diselesaikan dengan klastering, dan jelaskan mengapa label tidak tersedia pada kasus itu.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 2 — K-Means dengan Scikit-Learn

**Pertanyaan yang ingin dijawab:** seberapa andal K-Means, dan apa pengaruh pemilihan pusat awal?

### 2.1 Melatih K-Means

Lengkapi bagian bertanda `____`.

> **Perhatikan.** Argumen `n_init=1` sengaja dipakai agar algoritma **hanya dijalankan sekali** dari satu inisialisasi. Nilai bawaan scikit-learn adalah `n_init=10`, yaitu menjalankannya sepuluh kali lalu memilih hasil terbaik — dan itulah yang menyembunyikan masalah yang ingin kita amati.

In [ ]:
# TODO (a): buat model KMeans dengan 4 klaster, random_state=SEED,
#           n_init=1, dan init='random'
km = KMeans(n_clusters=____, random_state=SEED, n_init=1, init='random')

# TODO (b): latih model pada X_bulat.
#           PERHATIKAN: hanya X saja, TANPA y!
km.fit(____)

label_km = km.labels_
pusat_km = km.cluster_centers_

print("Inertia (jumlah kuadrat jarak ke pusat):", round(km.inertia_, 3))
print("Ukuran tiap klaster:", np.bincount(label_km))
print("\nPusat klaster:\n", np.round(pusat_km, 2))

### 2.2 Menggambar hasilnya

In [ ]:
plt.scatter(X_bulat[:, 0], X_bulat[:, 1], c=label_km, cmap="viridis", s=18, alpha=0.7)
plt.scatter(pusat_km[:, 0], pusat_km[:, 1], c="red", marker="X", s=250,
            edgecolor="black", label="Pusat klaster")
plt.title(f"Hasil K-Means (inertia = {km.inertia_:.1f})")
plt.xlabel("Fitur 1"); plt.ylabel("Fitur 2"); plt.legend()
plt.show()

### 2.3 Menguji kepekaan terhadap inisialisasi

Sel berikut menjalankan K-Means **sepuluh kali** dengan pusat awal yang berbeda-beda, lalu mencatat inertia masing-masing.

In [ ]:
hasil = []
for s in range(10):
    m = KMeans(n_clusters=4, random_state=SEED + s, n_init=1, init='random').fit(X_bulat)
    hasil.append({"percobaan": s + 1,
                  "random_state": SEED + s,
                  "inertia": round(m.inertia_, 3),
                  "ukuran_klaster": np.sort(np.bincount(m.labels_)).tolist()})

tabel_k2 = pd.DataFrame(hasil)
display(tabel_k2)

print("Inertia terkecil :", tabel_k2['inertia'].min())
print("Inertia terbesar :", tabel_k2['inertia'].max())
print("Selisih          :", round(tabel_k2['inertia'].max() - tabel_k2['inertia'].min(), 3))
print("Banyaknya nilai inertia yang berbeda:", tabel_k2['inertia'].nunique())

### 2.4 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| Inertia pada percobaan pertama | ...... |
| Inertia terkecil dari 10 percobaan | ...... |
| Inertia terbesar dari 10 percobaan | ...... |
| Selisih terbesar − terkecil | ...... |
| Banyaknya nilai inertia yang berbeda | ...... |

### 2.5 Pertanyaan Analisis

**B1.** Apakah kesepuluh percobaan menghasilkan inertia yang sama? Bila tidak, apa yang menyebabkan perbedaannya?

> *Jawaban Anda:*
>
> ......

**B2.** Inertia mana yang menandakan hasil pengelompokan **lebih baik**: yang besar atau yang kecil? Jelaskan berdasarkan definisi inertia.

> *Jawaban Anda:*
>
> ......

**B3.** Berdasarkan temuan Anda, mengapa scikit-learn memakai `n_init=10` sebagai nilai bawaan? Apa yang sebenarnya dikerjakan argumen itu?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 3 — Menuliskan Sendiri Algoritma Lloyd

**Pertanyaan yang ingin dijawab:** apa yang sebenarnya terjadi di dalam `KMeans.fit()`?

**Algoritma Lloyd terdiri atas tiga tahap:**

| Tahap | Yang dikerjakan |
|---|---|
| **Inisialisasi** | Pilih `k` titik data secara acak sebagai pusat awal |
| **Penugasan** | Tetapkan setiap titik ke pusat **terdekat** |
| **Pembaruan** | Geser setiap pusat ke **rata-rata** titik anggotanya |

Tahap penugasan dan pembaruan diulang bergantian sampai penugasannya tidak berubah lagi.

### 3.1 Melengkapi ketiga fungsi

Lengkapi bagian bertanda `____`.

In [ ]:
def inisialisasi_pusat(x, k, rng):
    """Metode Forgy: memilih k titik data berbeda secara acak sebagai pusat awal."""
    idx = rng.choice(len(x), size=k, replace=False)
    return x[idx]


def tugaskan(x, pusat):
    """Menetapkan setiap titik ke pusat terdekat.

    Mengembalikan array berisi nomor pusat terdekat untuk tiap titik.
    """
    # Broadcasting: menghitung selisih setiap titik terhadap setiap pusat sekaligus.
    # Bentuk hasilnya (jumlah_titik, jumlah_pusat, jumlah_fitur)
    selisih = x[:, np.newaxis] - pusat
    jarak = np.linalg.norm(selisih, axis=2)   # (jumlah_titik, jumlah_pusat)

    # TODO (a): ambil indeks pusat dengan jarak TERKECIL untuk setiap titik.
    #           Petunjuk: gunakan np.argmin pada sumbu yang tepat.
    return np.____(jarak, axis=1)


def perbarui_pusat(x, tugas, k):
    """Menggeser setiap pusat ke rata-rata titik anggotanya."""
    pusat_baru = []
    for j in range(k):
        anggota = x[tugas == j]
        if len(anggota) == 0:                 # klaster kosong: pusat diacak ulang
            pusat_baru.append(x[np.random.randint(len(x))])
        else:
            # TODO (b): hitung rata-rata seluruh anggota klaster ini.
            #           Petunjuk: rata-rata dihitung per kolom (per fitur).
            pusat_baru.append(anggota.____(axis=0))
    return np.array(pusat_baru)


print("Ketiga fungsi siap. Jalankan sel berikutnya untuk merangkainya.")

### 3.2 Merangkai menjadi algoritma lengkap

In [ ]:
def kmeans_manual(x, k, seed, maks_iterasi=100):
    """Algoritma Lloyd lengkap. Berhenti ketika penugasan tidak berubah lagi."""
    rng = np.random.default_rng(seed)
    pusat = inisialisasi_pusat(x, k, rng)
    tugas = tugaskan(x, pusat)
    riwayat = [pusat.copy()]

    for i in range(maks_iterasi):
        pusat = perbarui_pusat(x, tugas, k)
        tugas_baru = tugaskan(x, pusat)
        riwayat.append(pusat.copy())
        if np.all(tugas_baru == tugas):       # sudah konvergen
            tugas = tugas_baru
            break
        tugas = tugas_baru

    return pusat, tugas, riwayat, i + 1


pusat_manual, tugas_manual, riwayat, n_iterasi = kmeans_manual(X_bulat, 4, SEED)

# Inertia dihitung sendiri agar dapat dibandingkan dengan hasil sklearn
inertia_manual = sum(((X_bulat[tugas_manual == j] - pusat_manual[j]) ** 2).sum()
                     for j in range(4))

print("Konvergen setelah", n_iterasi, "iterasi")
print("Inertia versi manual :", round(inertia_manual, 3))
print("Ukuran tiap klaster  :", np.bincount(tugas_manual))

### 3.3 Membandingkan dengan hasil scikit-learn

In [ ]:
km_pembanding = KMeans(n_clusters=4, random_state=SEED, n_init=10).fit(X_bulat)

print("Inertia versi manual     :", round(inertia_manual, 3))
print("Inertia versi scikit-learn:", round(km_pembanding.inertia_, 3))
print("\nKesamaan pengelompokan (Adjusted Rand Index):",
      round(adjusted_rand_score(tugas_manual, km_pembanding.labels_), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(X_bulat[:, 0], X_bulat[:, 1], c=tugas_manual, cmap="viridis", s=18, alpha=0.7)
ax[0].scatter(pusat_manual[:, 0], pusat_manual[:, 1], c="red", marker="X", s=200, edgecolor="black")
ax[0].set_title("Implementasi manual")

ax[1].scatter(X_bulat[:, 0], X_bulat[:, 1], c=km_pembanding.labels_, cmap="viridis", s=18, alpha=0.7)
ax[1].scatter(km_pembanding.cluster_centers_[:, 0], km_pembanding.cluster_centers_[:, 1],
              c="red", marker="X", s=200, edgecolor="black")
ax[1].set_title("Scikit-Learn")
plt.tight_layout(); plt.show()

### 3.4 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| Jumlah iterasi sampai konvergen | ...... |
| Inertia versi manual | ...... |
| Inertia versi scikit-learn | ...... |
| Adjusted Rand Index antara keduanya | ...... |

### 3.5 Pertanyaan Analisis

**C1.** Apakah kedua nilai inertia sama atau berbeda? Bila berbeda, jelaskan penyebabnya (ingat kembali temuan Anda pada Kegiatan 2).

> *Jawaban Anda:*
>
> ......

**C2.** Adjusted Rand Index bernilai 1 bila kedua pengelompokan identik. Berapa nilai Anda, dan apa artinya?

> *Jawaban Anda:*
>
> ......

**C3.** Mengapa algoritma ini **dijamin berhenti**? Petunjuk: pikirkan tentang nilai inertia pada setiap langkah dan banyaknya kemungkinan penugasan.

> *Jawaban Anda:*
>
> ......

**C4.** Nomor klaster pada kedua panel bisa berbeda warnanya walaupun pengelompokannya sama persis. Mengapa penomoran klaster tidak punya makna intrinsik?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 4 — Memilih Jumlah Klaster

**Pertanyaan yang ingin dijawab:** dalam kasus nyata kita tidak tahu berapa jumlah kelompoknya. Bagaimana menentukannya?

Kita memakai dua alat:

| Alat | Yang diukur | Nilai baik |
|---|---|---|
| **Inertia** | Jumlah kuadrat jarak titik ke pusatnya | Kecil, tetapi **selalu** turun seiring k |
| **Silhouette** | Seberapa rapat klasternya dibanding jarak ke klaster lain | Mendekati 1 |

> **Kunci pemahamannya.** Inertia **selalu** menurun ketika `k` bertambah. Pada kasus ekstrem, bila `k` sama dengan jumlah titik data, setiap titik menjadi pusatnya sendiri dan inertia menjadi nol — pengelompokan yang sempurna sekaligus tidak berguna sama sekali. Karena itu kita mencari **titik siku**, bukan nilai terkecil.

### 4.1 Menghitung kedua ukuran

Lengkapi bagian bertanda `____`.

In [ ]:
daftar_k = range(2, 11)
inertias, silhouettes = [], []

for k in daftar_k:
    m = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_bulat)

    # TODO (a): ambil nilai inertia dari model m
    inertias.append(m.____)

    # TODO (b): hitung skor silhouette.
    #           Urutannya: silhouette_score(data, label_hasil_klastering)
    silhouettes.append(silhouette_score(X_bulat, m.____))

tabel_k4 = pd.DataFrame({"k": list(daftar_k),
                         "inertia": np.round(inertias, 2),
                         "silhouette": np.round(silhouettes, 4)})
tabel_k4["penurunan_inertia"] = tabel_k4["inertia"].diff().round(2)
display(tabel_k4)

print("k dengan silhouette tertinggi:", tabel_k4.loc[tabel_k4['silhouette'].idxmax(), 'k'])

### 4.2 Menggambar kedua kurva

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

ax[0].plot(list(daftar_k), inertias, "o-", lw=2)
ax[0].set_title("Metode Siku (Elbow)")
ax[0].set_xlabel("Jumlah klaster (k)"); ax[0].set_ylabel("Inertia")
ax[0].grid(alpha=0.3)

ax[1].plot(list(daftar_k), silhouettes, "o-", lw=2, color="darkorange")
k_terbaik = tabel_k4.loc[tabel_k4['silhouette'].idxmax(), 'k']
ax[1].axvline(k_terbaik, ls="--", c="red", label=f"k terbaik = {k_terbaik}")
ax[1].set_title("Skor Silhouette")
ax[1].set_xlabel("Jumlah klaster (k)"); ax[1].set_ylabel("Silhouette")
ax[1].legend(); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

### 4.3 Tabel Hasil Pengamatan

| k | Inertia | Penurunan inertia | Silhouette |
|---|---|---|---|
| 2 | ...... | — | ...... |
| 3 | ...... | ...... | ...... |
| 4 | ...... | ...... | ...... |
| 5 | ...... | ...... | ...... |
| 6 | ...... | ...... | ...... |

| Ringkasan | Nilai |
|---|---|
| Letak titik siku menurut Anda | k = ...... |
| k dengan silhouette tertinggi | k = ...... |
| Jumlah kelompok sebenarnya | k = ...... |

### 4.4 Pertanyaan Analisis

**D1.** Apakah inertia pernah **naik** ketika k bertambah? Jelaskan mengapa demikian.

> *Jawaban Anda:*
>
> ......

**D2.** Perhatikan kolom `penurunan_inertia`. Pada nilai k berapa penurunannya mulai jauh mengecil? Kaitkan dengan letak titik siku.

> *Jawaban Anda:*
>
> ......

**D3.** Apakah kedua metode (siku dan silhouette) menunjuk nilai k yang sama? Bila berbeda, mana yang akan Anda percaya dan mengapa?

> *Jawaban Anda:*
>
> ......

**D4.** Andaikan pada kasus nyata Anda tidak tahu jumlah kelompok sebenarnya. Selain kedua ukuran ini, faktor apa lagi yang perlu dipertimbangkan saat memilih k?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 5 — Gaussian Mixture Model (GMM)

**Pertanyaan yang ingin dijawab:** apa keunggulan GMM dibandingkan K-Means?

**Dua perbedaan pokoknya:**

| Aspek | K-Means | GMM |
|---|---|---|
| Penugasan | **Tegas** — satu titik satu klaster | **Probabilistik** — satu titik punya peluang untuk tiap komponen |
| Bentuk klaster | Selalu **bulat** (jarak Euclidean) | Boleh **elips miring** (matriks kovariansi) |
| Keluaran | `predict` saja | `predict` **dan** `predict_proba` |

Untuk melihat bedanya, kita memakai `X_elips`, yaitu data yang gerombolannya sengaja dimiringkan.

### 5.1 Membandingkan K-Means dan GMM pada data elips

Lengkapi bagian bertanda `____`.

In [ ]:
km_elips = KMeans(n_clusters=4, random_state=SEED, n_init=10).fit(X_elips)

# TODO (a): buat GaussianMixture dengan 4 komponen dan random_state=SEED
gmm_elips = GaussianMixture(n_components=____, random_state=SEED)

# TODO (b): latih GMM pada X_elips (sekali lagi: tanpa label!)
gmm_elips.fit(____)

label_gmm = gmm_elips.predict(X_elips)

ari_km  = adjusted_rand_score(y_asli, km_elips.labels_)
ari_gmm = adjusted_rand_score(y_asli, label_gmm)

print("Kesesuaian dengan kelompok sebenarnya (Adjusted Rand Index)")
print("  K-Means :", round(ari_km, 4))
print("  GMM     :", round(ari_gmm, 4))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].scatter(X_elips[:, 0], X_elips[:, 1], c=y_asli, cmap="viridis", s=16, alpha=0.7)
ax[0].set_title("Kelompok sebenarnya")
ax[1].scatter(X_elips[:, 0], X_elips[:, 1], c=km_elips.labels_, cmap="viridis", s=16, alpha=0.7)
ax[1].set_title(f"K-Means (ARI = {ari_km:.3f})")
ax[2].scatter(X_elips[:, 0], X_elips[:, 1], c=label_gmm, cmap="viridis", s=16, alpha=0.7)
ax[2].set_title(f"GMM (ARI = {ari_gmm:.3f})")
plt.tight_layout(); plt.show()

### 5.2 Soft Clustering: Melihat Tingkat Keyakinan

Inilah yang tidak dapat diberikan K-Means sama sekali.

> **Kita berpindah dataset.** Pada `X_elips` keempat gerombolannya terpisah sangat jelas, sehingga GMM hampir selalu yakin 100% — tidak ada yang menarik untuk diamati. Karena itu bagian ini memakai `X_tumpang`, yaitu tiga gerombolan yang **saling bertumpang tindih**, agar keraguan model benar-benar terlihat.

In [ ]:
gmm_tumpang = GaussianMixture(n_components=3, random_state=SEED).fit(X_tumpang)

# TODO (c): ambil probabilitas keanggotaan setiap titik untuk tiap komponen.
#           Petunjuk: metodenya mirip predict_proba pada regresi logistik.
proba = gmm_tumpang.____(X_tumpang)

keyakinan = proba.max(axis=1)     # probabilitas tertinggi untuk tiap titik

print("Bentuk matriks probabilitas:", proba.shape, " (baris = titik, kolom = komponen)")
print("Setiap baris berjumlah 1?", np.allclose(proba.sum(axis=1), 1))
print("\nKeyakinan rata-rata :", round(keyakinan.mean(), 4))
print("Keyakinan terendah  :", round(keyakinan.min(), 4))
print("Jumlah titik dengan keyakinan < 0,60 :", int((keyakinan < 0.60).sum()))
print("Jumlah titik dengan keyakinan < 0,80 :", int((keyakinan < 0.80).sum()))

print("\nTiga titik yang paling meragukan bagi model:")
display(pd.DataFrame(np.round(proba[np.argsort(keyakinan)[:3]], 3),
                     columns=["Komponen 0", "Komponen 1", "Komponen 2"]))

plt.scatter(X_tumpang[:, 0], X_tumpang[:, 1], c=keyakinan, cmap="coolwarm_r",
            s=26, alpha=0.85)
plt.colorbar(label="Keyakinan model (probabilitas tertinggi)")
plt.title("Titik biru = model yakin, titik merah = model ragu")
plt.xlabel("Fitur 1"); plt.ylabel("Fitur 2")
plt.show()

### 5.3 Memilih jumlah komponen dengan BIC

GMM menyediakan ukuran yang tidak dimiliki K-Means: **BIC** (*Bayesian Information Criterion*). Ukuran ini menghukum model yang terlalu rumit, sehingga **boleh dicari nilai terkecilnya** — berbeda dengan inertia.

In [ ]:
bics = []
for k in range(2, 11):
    g = GaussianMixture(n_components=k, random_state=SEED).fit(X_elips)
    bics.append(g.bic(X_elips))

tabel_bic = pd.DataFrame({"k": list(range(2, 11)), "BIC": np.round(bics, 1)})
display(tabel_bic)

k_bic = tabel_bic.loc[tabel_bic['BIC'].idxmin(), 'k']
print("k dengan BIC terkecil:", k_bic)

plt.plot(range(2, 11), bics, "o-", lw=2, color="seagreen")
plt.axvline(k_bic, ls="--", c="red", label=f"BIC terkecil pada k = {k_bic}")
plt.title("BIC terhadap Jumlah Komponen")
plt.xlabel("Jumlah komponen (k)"); plt.ylabel("BIC")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 5.4 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| ARI K-Means pada data elips | ...... |
| ARI GMM pada data elips | ...... |
| Selisih ARI (GMM − K-Means) | ...... |
| Keyakinan rata-rata GMM (pada `X_tumpang`) | ...... |
| Keyakinan terendah | ...... |
| Jumlah titik dengan keyakinan < 0,60 | ...... |
| Jumlah titik dengan keyakinan < 0,80 | ...... |
| k dengan BIC terkecil | ...... |

### 5.5 Pertanyaan Analisis

**E1.** Model mana yang lebih sesuai dengan kelompok sebenarnya pada data elips? Jelaskan penyebabnya dengan mengaitkan bentuk klaster dan jenis jarak yang dipakai masing-masing.

> *Jawaban Anda:*
>
> ......

**E2.** Pada grafik keyakinan (sel 5.2), di **posisi seperti apa** titik-titik berwarna merah berada? Mengapa masuk akal bahwa model ragu di posisi itu? Kaitkan juga dengan tabel tiga titik paling meragukan — bagaimana bentuk sebaran probabilitasnya?

> *Jawaban Anda:*
>
> ......

**E3.** Sebutkan satu situasi nyata ketika mengetahui **tingkat keyakinan** jauh lebih berguna daripada sekadar mengetahui nomor klaster.

> *Jawaban Anda:*
>
> ......

**E4.** Mengapa BIC boleh dicari nilai **terkecilnya**, sedangkan inertia tidak boleh diperlakukan begitu? Apa yang membuat keduanya berbeda?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 6 — Probabilitas dan Teorema Bayes

**Pertanyaan yang ingin dijawab:** bila sebuah detektor berbunyi, seberapa besar kemungkinan yang dideteksinya memang benar terjadi?

**Teorema Bayes:**

$$
P(W = 1 \mid D = 1) = \frac{P(D=1 \mid W=1)\,P(W=1)}{P(D=1 \mid W=1)\,P(W=1) + P(D=1 \mid W=0)\,P(W=0)}
$$

| Simbol | Arti | Istilah |
|---|---|---|
| $P(W=1)$ | Peluang kejadiannya memang terjadi | **prior** |
| $P(D=1 \mid W=1)$ | Peluang detektor berbunyi saat kejadian benar terjadi | **recall** |
| $P(D=1 \mid W=0)$ | Peluang detektor berbunyi padahal tidak terjadi apa-apa | **false positive rate** |
| $P(W=1 \mid D=1)$ | Yang ingin dicari | **posterior** |

Penyebutnya berasal dari **hukum probabilitas total**: detektor dapat berbunyi karena dua sebab — kejadiannya memang terjadi, atau detektornya keliru.

### 6.1 Menuliskan fungsinya

Lengkapi bagian bertanda `____`.

In [ ]:
def bayes(prior, recall, fpr):
    """Menghitung P(W=1 | D=1) dengan Teorema Bayes.

    Argumen:
        prior : P(W=1), peluang kejadian itu terjadi
        recall: P(D=1 | W=1), peluang deteksi benar
        fpr   : P(D=1 | W=0), peluang deteksi palsu
    """
    # TODO (a): pembilangnya adalah recall dikali prior
    pembilang = ____ * ____

    # TODO (b): penyebutnya memakai hukum probabilitas total.
    #           Lengkapi suku keduanya: fpr dikali peluang kejadian TIDAK terjadi.
    penyebut = recall * prior + fpr * (____)

    return pembilang / penyebut


# Uji cepat: detektor sangat sensitif, kejadiannya sangat jarang
print("Uji cepat:", round(bayes(prior=0.0001, recall=0.99, fpr=0.001), 5))

### 6.2 Pengaruh false positive rate dan recall

Sekarang kita amati mana yang lebih menentukan: menaikkan recall, atau menurunkan false positive rate.

In [ ]:
prior_tetap = 0.0001     # kejadian sangat jarang: 1 dari 10.000

# Percobaan 1: recall sempurna, fpr divariasikan
fprs = np.logspace(-6, -1, 60)
hasil_fpr = [bayes(prior_tetap, recall=1.0, fpr=f) for f in fprs]

# Percobaan 2: fpr tetap, recall divariasikan
recalls = np.linspace(0.01, 1.0, 60)
hasil_recall = [bayes(prior_tetap, recall=r, fpr=0.001) for r in recalls]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].semilogx(fprs, hasil_fpr, lw=2)
ax[0].set_title("Recall sempurna (=1,0)")
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("P(W=1 | D=1)")
ax[0].grid(alpha=0.3)

ax[1].plot(recalls, hasil_recall, lw=2, color="darkorange")
ax[1].set_title("False positive rate tetap (=0,001)")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("P(W=1 | D=1)")
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Recall 1,00 dengan fpr 0,001   -> P(W=1|D=1) =",
      round(bayes(prior_tetap, 1.00, 0.001), 5))
print("Recall 0,50 dengan fpr 0,001   -> P(W=1|D=1) =",
      round(bayes(prior_tetap, 0.50, 0.001), 5))
print("Recall 1,00 dengan fpr 0,00001 -> P(W=1|D=1) =",
      round(bayes(prior_tetap, 1.00, 0.00001), 5))

### 6.3 Pengaruh nilai prior

In [ ]:
priors = [0.0001, 0.001, 0.01, 0.1, 0.5]
tabel_prior = pd.DataFrame({
    "prior": priors,
    "P(W=1|D=1)": [round(bayes(p, recall=0.99, fpr=0.01), 5) for p in priors]
})
display(tabel_prior)

print("Seluruh baris memakai detektor yang SAMA: recall 99%, false positive rate 1%.")

### 6.4 Tabel Hasil Pengamatan

| Skenario (prior = 0,0001) | P(W=1 \| D=1) |
|---|---|
| Recall 1,00 · fpr 0,001 | ...... |
| Recall 0,50 · fpr 0,001 | ...... |
| Recall 1,00 · fpr 0,00001 | ...... |

| prior | P(W=1 \| D=1) dengan recall 0,99 dan fpr 0,01 |
|---|---|
| 0,0001 | ...... |
| 0,001 | ...... |
| 0,01 | ...... |
| 0,1 | ...... |
| 0,5 | ...... |

### 6.5 Pertanyaan Analisis

**F1.** Bandingkan ketiga baris pada tabel pertama. Mana yang lebih menaikkan $P(W=1 \mid D=1)$: menaikkan recall dari 0,50 menjadi 1,00, atau menurunkan fpr dari 0,001 menjadi 0,00001? Sebutkan angkanya.

> *Jawaban Anda:*
>
> ......

**F2.** Jelaskan **mengapa** demikian. Petunjuk: pada kejadian yang sangat jarang, berapa banyak kesempatan detektor untuk salah dibandingkan kesempatannya untuk benar?

> *Jawaban Anda:*
>
> ......

**F3.** Sebuah tes penyakit memiliki recall 99% dan false positive rate 1%. Penyakit itu menjangkiti 1 dari 10.000 orang. Bila hasil tes seseorang positif, berapa peluang ia benar-benar sakit? Hitung dengan fungsi `bayes` dan jelaskan mengapa hasilnya mengejutkan.

> *Jawaban Anda:*
>
> ......

**F4.** Sebutkan **dua** contoh sistem deteksi kejadian langka di dunia nyata, dan jelaskan pelajaran apa yang dapat diambil dari temuan Anda pada kegiatan ini.

> *Jawaban Anda:*
>
> ......


---
# Kesimpulan dan Refleksi

### Kesimpulan

*Tuliskan minimal enam poin kesimpulan berdasarkan hasil percobaan Anda sendiri. Sebutkan angka bila relevan, dan hindari kalimat teori umum yang bisa ditulis tanpa menjalankan kode.*

1. ......
2. ......
3. ......
4. ......
5. ......
6. ......

### Tabel Perbandingan K-Means dan GMM

*Isi berdasarkan pengalaman Anda mengerjakan Kegiatan 2 sampai 5.*

| Aspek | K-Means | GMM |
|---|---|---|
| Jenis penugasan | ...... | ...... |
| Bentuk klaster yang bisa ditangani | ...... | ...... |
| Ukuran untuk memilih k | ...... | ...... |
| Keluaran tambahan | ...... | ...... |
| ARI pada data elips (percobaan Anda) | ...... | ...... |
| Kapan sebaiknya dipakai | ...... | ...... |

### Refleksi

**R1.** Bagian mana dari praktikum ini yang paling sulit Anda pahami? Apa yang akhirnya membuat Anda mengerti, atau apa yang masih mengganjal?

> *Jawaban Anda:*
>
> ......

**R2.** Error apa yang Anda temui saat mengerjakan, dan bagaimana Anda mengatasinya? Sebutkan minimal satu.

> *Jawaban Anda:*
>
> ......

**R3.** Pada Pertemuan 03 kita mengerjakan *supervised learning*, pada pertemuan ini *unsupervised*. Menurut Anda, bagian mana dari alur kerjanya yang **tetap sama**, dan bagian mana yang **paling berubah**?

> *Jawaban Anda:*
>
> ......


---
# Rubrik Penilaian

*Bagian ini diisi oleh dosen atau asisten praktikum.*

| No | Aspek yang Dinilai | Bobot | Skor (0–100) | Nilai |
|---|---|---|---|---|
| 1 | Kelengkapan tabel hasil pengamatan (Kegiatan 1–6) | 20% | | |
| 2 | Ketepatan pengisian kode bertanda `TODO` | 25% | | |
| 3 | Kualitas jawaban analisis (A1–F4) | 30% | | |
| 4 | Tabel perbandingan K-Means vs GMM | 10% | | |
| 5 | Kesimpulan dan refleksi | 15% | | |
| | **Nilai Akhir** | **100%** | | |

**Catatan dosen:**

> ......

### Pedoman skor jawaban analisis

| Skor | Kriteria |
|---|---|
| 85–100 | Jawaban tepat, didukung angka dari percobaan sendiri, dan menunjukkan penalaran yang jelas |
| 70–84 | Jawaban tepat tetapi kurang didukung data atau penalarannya dangkal |
| 55–69 | Jawaban sebagian benar, atau hanya mengulang teori tanpa mengaitkan hasil percobaan |
| < 55 | Jawaban tidak tepat, kosong, atau merupakan salinan dari mahasiswa lain |


---
# Daftar Periksa Sebelum Mengumpulkan

Centang dengan mengganti `[ ]` menjadi `[x]` (klik dua kali sel ini untuk mengedit).

- [ ] Identitas pada bagian atas sudah diisi lengkap
- [ ] `NIM_3_DIGIT` sudah diganti dengan NIM saya sendiri
- [ ] Semua sel bertanda `TODO` sudah dilengkapi dan berjalan tanpa error
- [ ] Seluruh sel sudah dijalankan berurutan dari atas ke bawah
- [ ] Semua tabel hasil pengamatan sudah diisi angka dari komputer saya
- [ ] Semua pertanyaan analisis A1 sampai F4 sudah dijawab
- [ ] Tabel perbandingan K-Means vs GMM sudah lengkap
- [ ] Kesimpulan minimal enam poin dan refleksi R1–R3 sudah ditulis
- [ ] Notebook sudah disimpan (`Ctrl + S`) sebelum diekspor

### Langkah ekspor PDF

1. Simpan notebook: `Ctrl + S`
2. `File` → `Save and Export Notebook As...` → `HTML`
3. Buka berkas HTML di browser, tekan `Ctrl + P`, pilih **Save as PDF**
4. Pada dialog cetak, aktifkan **Background graphics** agar grafik ikut tercetak berwarna
5. Beri nama berkas: `LKM04_NIM_NamaLengkap.pdf`

---

*Lembar Kerja Mahasiswa — Praktikum Pertemuan 04*
*Materi diadaptasi dari kuliah "Machine Learning Mechanics" oleh Joseph E. Gonzalez dan Narges Norouzi.*
